In [1]:
import json
import socket
import logging
import time
import ssl
from threading import Thread

# set to true on debug environment only
DEBUG = True

#default connection properites
DEFAULT_XAPI_ADDRESS        = 'xapia.x-station.eu'
DEFAULT_XAPI_PORT           = 5124
DEFUALT_XAPI_STREAMING_PORT = 5125

# wrapper name and version
WRAPPER_NAME    = 'python'
WRAPPER_VERSION = '2.5.0'

# API inter-command timeout (in ms)
API_SEND_TIMEOUT = 100

# max connection tries
API_MAX_CONN_TRIES = 3

# logger properties
logger = logging.getLogger("jsonSocket")
FORMAT = '[%(asctime)-15s][%(funcName)s:%(lineno)d] %(message)s'
logging.basicConfig(format=FORMAT)

if DEBUG:
    logger.setLevel(logging.DEBUG)
else:
    logger.setLevel(logging.CRITICAL)


class TransactionSide(object):
    BUY = 0
    SELL = 1
    BUY_LIMIT = 2
    SELL_LIMIT = 3
    BUY_STOP = 4
    SELL_STOP = 5
    
class TransactionType(object):
    ORDER_OPEN = 0
    ORDER_CLOSE = 2
    ORDER_MODIFY = 3
    ORDER_DELETE = 4

class JsonSocket(object):
    def __init__(self, address, port, encrypt = False):
        self._ssl = encrypt 
        if self._ssl != True:
            self.socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        else:
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            self.socket = ssl.wrap_socket(sock)
        self.conn = self.socket
        self._timeout = None
        self._address = address
        self._port = port
        self._decoder = json.JSONDecoder()
        self._receivedData = ''

    def connect(self):
        for i in range(API_MAX_CONN_TRIES):
            try:
                self.socket.connect( (self.address, self.port) )
            except socket.error as msg:
                logger.error("SockThread Error: %s" % msg)
                time.sleep(0.25);
                continue
            logger.info("Socket connected")
            return True
        return False

    def _sendObj(self, obj):
        msg = json.dumps(obj)
        self._waitingSend(msg)

    def _waitingSend(self, msg):
        if self.socket:
            sent = 0
            msg = msg.encode('utf-8')
            while sent < len(msg):
                sent += self.conn.send(msg[sent:])
                logger.info('Sent: ' + str(msg))
                time.sleep(API_SEND_TIMEOUT/1000)

    def _read(self, bytesSize=4096):
        if not self.socket:
            raise RuntimeError("socket connection broken")
        while True:
            char = self.conn.recv(bytesSize).decode()
            self._receivedData += char
            try:
                (resp, size) = self._decoder.raw_decode(self._receivedData)
                if size == len(self._receivedData):
                    self._receivedData = ''
                    break
                elif size < len(self._receivedData):
                    self._receivedData = self._receivedData[size:].strip()
                    break
            except ValueError as e:
                continue
        logger.info('Received: ' + str(resp))
        return resp

    def _readObj(self):
        msg = self._read()
        return msg

    def close(self):
        logger.debug("Closing socket")
        self._closeSocket()
        if self.socket is not self.conn:
            logger.debug("Closing connection socket")
            self._closeConnection()

    def _closeSocket(self):
        self.socket.close()

    def _closeConnection(self):
        self.conn.close()

    def _get_timeout(self):
        return self._timeout

    def _set_timeout(self, timeout):
        self._timeout = timeout
        self.socket.settimeout(timeout)

    def _get_address(self):
        return self._address

    def _set_address(self, address):
        pass

    def _get_port(self):
        return self._port

    def _set_port(self, port):
        pass

    def _get_encrypt(self):
        return self._ssl

    def _set_encrypt(self, encrypt):
        pass

    timeout = property(_get_timeout, _set_timeout, doc='Get/set the socket timeout')
    address = property(_get_address, _set_address, doc='read only property socket address')
    port = property(_get_port, _set_port, doc='read only property socket port')
    encrypt = property(_get_encrypt, _set_encrypt, doc='read only property socket port')
    
    
class APIClient(JsonSocket):
    def __init__(self, address=DEFAULT_XAPI_ADDRESS, port=DEFAULT_XAPI_PORT, encrypt=True):
        super(APIClient, self).__init__(address, port, encrypt)
        if(not self.connect()):
            raise Exception("Cannot connect to " + address + ":" + str(port) + " after " + str(API_MAX_CONN_TRIES) + " retries")

    def execute(self, dictionary):
        self._sendObj(dictionary)
        return self._readObj()    

    def disconnect(self):
        self.close()
        
    def commandExecute(self,commandName, arguments=None):
        return self.execute(baseCommand(commandName, arguments))

class APIStreamClient(JsonSocket):
    def __init__(self, address=DEFAULT_XAPI_ADDRESS, port=DEFUALT_XAPI_STREAMING_PORT, encrypt=True, ssId=None, 
                 tickFun=None, tradeFun=None, balanceFun=None, tradeStatusFun=None, profitFun=None, newsFun=None):
        super(APIStreamClient, self).__init__(address, port, encrypt)
        self._ssId = ssId

        self._tickFun = tickFun
        self._tradeFun = tradeFun
        self._balanceFun = balanceFun
        self._tradeStatusFun = tradeStatusFun
        self._profitFun = profitFun
        self._newsFun = newsFun
        
        if(not self.connect()):
            raise Exception("Cannot connect to streaming on " + address + ":" + str(port) + " after " + str(API_MAX_CONN_TRIES) + " retries")

        self._running = True
        self._t = Thread(target=self._readStream, args=())
        self._t.setDaemon(True)
        self._t.start()

    def _readStream(self):
        while (self._running):
                msg = self._readObj()
                logger.info("Stream received: " + str(msg))
                if (msg["command"]=='tickPrices'):
                    self._tickFun(msg)
                elif (msg["command"]=='trade'):
                    self._tradeFun(msg)
                elif (msg["command"]=="balance"):
                    self._balanceFun(msg)
                elif (msg["command"]=="tradeStatus"):
                    self._tradeStatusFun(msg)
                elif (msg["command"]=="profit"):
                    self._profitFun(msg)
                elif (msg["command"]=="news"):
                    self._newsFun(msg)
    
    def disconnect(self):
        self._running = False
        self._t.join()
        self.close()

    def execute(self, dictionary):
        self._sendObj(dictionary)

    def subscribePrice(self, symbol):
        self.execute(dict(command='getTickPrices', symbol=symbol, streamSessionId=self._ssId))
        
    def subscribePrices(self, symbols):
        for symbolX in symbols:
            self.subscribePrice(symbolX)
    
    def subscribeTrades(self):
        self.execute(dict(command='getTrades', streamSessionId=self._ssId))
        
    def subscribeBalance(self):
        self.execute(dict(command='getBalance', streamSessionId=self._ssId))

    def subscribeTradeStatus(self):
        self.execute(dict(command='getTradeStatus', streamSessionId=self._ssId))

    def subscribeProfits(self):
        self.execute(dict(command='getProfits', streamSessionId=self._ssId))

    def subscribeNews(self):
        self.execute(dict(command='getNews', streamSessionId=self._ssId))


    def unsubscribePrice(self, symbol):
        self.execute(dict(command='stopTickPrices', symbol=symbol, streamSessionId=self._ssId))
        
    def unsubscribePrices(self, symbols):
        for symbolX in symbols:
            self.unsubscribePrice(symbolX)
    
    def unsubscribeTrades(self):
        self.execute(dict(command='stopTrades', streamSessionId=self._ssId))
        
    def unsubscribeBalance(self):
        self.execute(dict(command='stopBalance', streamSessionId=self._ssId))

    def unsubscribeTradeStatus(self):
        self.execute(dict(command='stopTradeStatus', streamSessionId=self._ssId))

    def unsubscribeProfits(self):
        self.execute(dict(command='stopProfits', streamSessionId=self._ssId))

    def unsubscribeNews(self):
        self.execute(dict(command='stopNews', streamSessionId=self._ssId))


# Command templates
def baseCommand(commandName, arguments=None):
    if arguments==None:
        arguments = dict()
    return dict([('command', commandName), ('arguments', arguments)])

def loginCommand(userId, password, appName=''):
    return baseCommand('login', dict(userId=userId, password=password, appName=appName))



# example function for processing ticks from Streaming socket
def procTickExample(msg): 
    print("TICK: ", msg)

# example function for processing trades from Streaming socket
def procTradeExample(msg): 
    print("TRADE: ", msg)

# example function for processing trades from Streaming socket
def procBalanceExample(msg): 
    print("BALANCE: ", msg)

# example function for processing trades from Streaming socket
def procTradeStatusExample(msg): 
    print("TRADE STATUS: ", msg)

# example function for processing trades from Streaming socket
def procProfitExample(msg): 
    print("PROFIT: ", msg)

# example function for processing news from Streaming socket
def procNewsExample(msg): 
    print("NEWS: ", msg)


In [3]:
#!/usr/bin/env python
import time
import logging

# Import necessary XTB API wrappers
# from xtb_wrapper import APIClient, loginCommand, TransactionSide, TransactionType

# Configure logging
logger = logging.getLogger("xtbTrade")
logger.setLevel(logging.INFO)

class XTBTrader:
    def __init__(self, user_id, password):
        self.client = None
        self.user_id = user_id
        self.password = password

    def connect(self):
        """Establish connection and log in."""
        self.client = APIClient()
        response = self.client.execute(loginCommand(self.user_id, self.password))
        if response.get("status"):
            logger.info("✅ Logged in successfully!")
            self.ssid = response.get("streamSessionId")
            logger.info("🔗 Stream session ID: %s", self.ssid)
        else:
            logger.error("❌ Login failed: %s", response)
            return False
        return True

    def place_trade(self, symbol, volume, price):
        """Place a market order."""
        trade_info = {
            "cmd": TransactionSide.BUY,
            "customComment": "Open trade test",
            "expiration": 0,
            "offset": 0,
            "order": 0,  # For new orders, this is set to 0
            "price": price,
            "sl": 0.0,
            "symbol": symbol,
            "tp": 0.0,
            "type": TransactionType.ORDER_OPEN,
            "volume": volume
        }
        response = self.client.commandExecute("tradeTransaction", {"tradeTransInfo": trade_info})
        if response.get("status"):
            order_number = response["returnData"]["order"]
            logger.info("✅ Trade placed successfully, order number: %s", order_number)
            return order_number
        else:
            logger.error("❌ Trade placement failed: %s", response)
            return None

    def check_trade_status(self, order):
        """Check the trade status."""
        response = self.client.commandExecute("tradeTransactionStatus", {"order": order})
        logger.info("📊 Trade status: %s", response)
        return response

    def close_trade(self, order, symbol, volume, price=None):
        """
        Close an existing trade using the tradeTransaction command with ORDER_CLOSE.
        Here we try using the order number + 1 for closing the trade.
        According to some reports, XTB may expect the close command's 'order' field to be the open order number plus one.
        If no price is provided, we also attempt to use the current bid price.
        """
        # If no price is provided, retrieve the current bid from the trade status.
        if price is None:
            status = self.check_trade_status(order)
            price = status["returnData"].get("bid", 0.0)
            logger.info("Using current bid price %s for closing trade", price)

        # Use order number + 1 for closing, as suggested.
        close_order = order + 1
        logger.info("Using order number %s (open order %s + 1) for closing", close_order, order)

        trade_info = {
            "cmd": TransactionSide.BUY,  # As per tutorial, use BUY to close a BUY trade.
            "customComment": "Close trade test",
            "expiration": 0,
            "offset": 0,
            "order": close_order,       # Use order number + 1 as recommended.
            "price": price,
            "sl": 0.0,
            "symbol": symbol,
            "tp": 0.0,
            "type": TransactionType.ORDER_CLOSE,
            "volume": volume
        }
        logger.info("Sending close trade command with: %s", trade_info)
        response = self.client.commandExecute("tradeTransaction", {"tradeTransInfo": trade_info})
        return response

    def disconnect(self):
        """Disconnect from API."""
        if self.client:
            self.client.disconnect()
            logger.info("🔌 Disconnected from XTB API.")

def main():
    """Main execution function."""
    USER_ID = 17535100
    PASSWORD = "GuiZarHoh2711!"

    SYMBOL = "EURUSD"
    VOLUME = 0.1
    OPEN_PRICE = 1.000            # Hypothetical open price; in live scenarios, use market prices

    # For closing, we intentionally leave price as None so that the script will use the current bid.
    trader = XTBTrader(USER_ID, PASSWORD)
    if not trader.connect():
        return

    order_number = trader.place_trade(SYMBOL, VOLUME, OPEN_PRICE)
    if not order_number:
        trader.disconnect()
        return

    # Allow some time for the trade to be processed
    time.sleep(2)
    trader.check_trade_status(order_number)

    # Wait before closing the trade (for demonstration)
    time.sleep(2)
    close_resp = trader.close_trade(order_number, SYMBOL, VOLUME)
    if close_resp.get("status"):
        logger.info("✅ Trade closed successfully, order: %s", order_number)
    else:
        logger.error("❌ Trade close failed: %s", close_resp)

    trader.disconnect()

if __name__ == "__main__":
    main()


/tmp/ipykernel_205/2709462794.py:58: DeprecationWarning: ssl.wrap_socket() is deprecated, use SSLContext.wrap_socket()
  self.socket = ssl.wrap_socket(sock)
[2025-02-28 11:07:29,012][connect:74] Socket connected
[2025-02-28 11:07:29,013][_waitingSend:88] Sent: b'{"command": "login", "arguments": {"userId": 17535100, "password": "GuiZarHoh2711!", "appName": ""}}'
[2025-02-28 11:07:29,262][_read:107] Received: {'status': True, 'streamSessionId': '005056fffeb96ad7-0020e9c1-007e1e65-ecaea3ab5d4b59ac-d3daabaf'}
[2025-02-28 11:07:29,262][connect:23] ✅ Logged in successfully!
[2025-02-28 11:07:29,263][connect:25] 🔗 Stream session ID: 005056fffeb96ad7-0020e9c1-007e1e65-ecaea3ab5d4b59ac-d3daabaf
[2025-02-28 11:07:29,264][_waitingSend:88] Sent: b'{"command": "tradeTransaction", "arguments": {"tradeTransInfo": {"cmd": 0, "customComment": "Open trade test", "expiration": 0, "offset": 0, "order": 0, "price": 1.0, "sl": 0.0, "symbol": "EURUSD", "tp": 0.0, "type": 0, "volume": 0.1}}}'
[2025-02-28 11: